# Signal Filtering 1 — See & Quantify the Noise
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

A noisy **weight / load-cell** signal: fill/rest/empty cycles with quantization dither, agitation/sloshing, and washing-fluid spikes. numpy, scipy, pandas and matplotlib are pre-installed in Colab.

> Runs on a synthetic signal that mimics a real load cell. **To use your own export**, replace the load cell with `pd.read_csv("your_export.csv", ...)` (two columns: timestamp, value).

## Objectives
By the end you will be able to:
- Plot an irregular sensor signal and zoom into the noise.
- Find the quantization step and the inter-sample timing.
- Measure at-rest noise sigma and read the residual power spectrum.

## Load

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
s = pd.read_csv(DATA + "weight_noisy.csv", parse_dates=["timestamp"]).set_index("timestamp")["weight"]
print("samples:", s.size, "| range: %.1f .. %.1f" % (s.min(), s.max()))
s.head()

## 1 · Look at the whole signal, then zoom in
The big moves are real (fills/empties). The fuzz is the noise we want to remove.

In [ ]:
plt.figure(figsize=(11,3)); plt.plot(s.index, s.values, lw=.5)
plt.title("Raw weight signal - fill/rest/empty cycles + noise"); plt.ylabel("weight"); plt.show()

zoom = s.iloc[2000:2300]
plt.figure(figsize=(11,3)); plt.plot(zoom.index, zoom.values, marker=".", ms=3, lw=.5)
plt.title("Zoom: dither between quantization levels while at rest"); plt.show()

## 2 · Quantization step & sampling
How finely is the value stored, and how regular are the timestamps?

In [ ]:
uv = np.unique(np.round(s.values, 6)); dq = np.diff(uv); dq = dq[dq > 0]
q = dq.min()
print("quantization step:", round(q, 4))

dt = np.diff(s.index.values).astype("timedelta64[s]").astype(float)
print("inter-sample dt  median %.1fs  mean %.1fs  max %.1fs" % (np.median(dt), dt.mean(), dt.max()))

## 3 · At-rest noise
Raw std is useless here - it is dominated by the real fills. Measure noise only where the signal is *flat*.

In [ ]:
med = s.rolling(15, center=True, min_periods=1).median()
local_range = (med.rolling(31, center=True, min_periods=1).max()
               - med.rolling(31, center=True, min_periods=1).min())
rest = local_range < 3 * q
resid = s - med
sigma_rest = resid[rest].std()
print("fraction of time at rest : %.1f%%" % (100 * rest.mean()))
print("at-rest noise sigma      : %.4f  (~%.1f quantization levels)" % (sigma_rest, sigma_rest / q))

## 4 · What kind of noise? (power spectrum)
The residual (signal minus local median) is resampled to a uniform 1 s grid and its power spectrum computed. A flat spectrum = broadband/random noise; a sharp peak = a periodic source (here, the ~8 s sloshing).

In [ ]:
from scipy.signal import welch
r = resid.resample("1s").mean().fillna(0).to_numpy(float)
r = r - r.mean()
f, P = welch(r, fs=1.0, nperseg=min(len(r), 2048))
plt.figure(figsize=(8,3)); plt.semilogy(f[1:], P[1:])
plt.title("Residual power spectrum"); plt.xlabel("Hz"); plt.ylabel("power"); plt.show()
flat = np.exp(np.mean(np.log(P[1:] + 1e-30))) / (P[1:].mean() + 1e-30)
print("spectral flatness: %.2f  (1.0 = white/broadband, ->0 = strongly periodic)" % flat)

## Debrief
- Three noise types are visible: fast **dither** (quantization), periodic **sloshing** (~8 s), and rare **spikes** (washing fluid). Each wants a different filter.
- The at-rest sigma is your target: a good filter should shrink it without lagging the real fills.
- Next (Filtering 2 & 3): actually remove the noise and measure how well.